# 05 — Review, clarification, and follow-on behavior

This notebook demonstrates:

- clarification flow,
- active-task redirection (`task redirect`) vs follow-on task creation,
- prototype artifact isolation in workspace paths,
- why prototype deliveries are blocked from learning/exemplar promotion.


In [ ]:
from __future__ import annotations

import json
import os
import shlex
import subprocess
import tempfile
from pathlib import Path
from textwrap import dedent


def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'pyproject.toml').exists():
            return candidate
    raise RuntimeError('Could not find repository root from current working directory.')


REPO_ROOT = find_repo_root(Path.cwd())
WORK_ROOT = Path(tempfile.mkdtemp(prefix='meridian_expert_demo_'))
WORKSPACE = WORK_ROOT / 'runtime'

os.environ['PYTHONPATH'] = str(REPO_ROOT / 'src')
os.environ['MERIDIAN_EXPERT_WORKSPACE'] = str(WORKSPACE)
os.environ['MERIDIAN_EXPERT_LLM_BACKEND'] = 'fake'
os.environ.setdefault('OPENAI_API_KEY', 'demo-not-used-with-fake-backend')

from meridian_expert.testing_support.repo_fixtures import build_fixture_workspace

repos = build_fixture_workspace(WORK_ROOT)
os.environ['MERIDIAN_REPO_PATH'] = str(repos['meridian'])
os.environ['MERIDIAN_AUX_REPO_PATH'] = str(repos['meridian_aux'])


def run_cli(args: str, check: bool = True) -> subprocess.CompletedProcess[str]:
    cmd = ['python', '-m', 'meridian_expert.cli', *shlex.split(args)]
    result = subprocess.run(cmd, cwd=REPO_ROOT, env=os.environ.copy(), text=True, capture_output=True)
    print('$', ' '.join(cmd))
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.stderr.strip():
        print('--- stderr ---')
        print(result.stderr.strip())
    if check and result.returncode != 0:
        raise RuntimeError(f'Command failed with code {result.returncode}: {args}')
    return result


def write_demo_task(relative_path: str, body: str) -> Path:
    path = WORK_ROOT / relative_path
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(dedent(body).strip() + '
', encoding='utf-8')
    return path


def read_text(path: Path, max_chars: int = 1600) -> None:
    text = path.read_text(encoding='utf-8')
    print(text[:max_chars] + ('
... (truncated)' if len(text) > max_chars else ''))


print('Repo root:', REPO_ROOT)
print('Demo workspace:', WORK_ROOT)
print('Meridian fixture:', repos['meridian'])
print('Meridian_aux fixture:', repos['meridian_aux'])


## Clarification flow

In [ ]:
ambiguous_task = write_demo_task('tasks/clarification_notebook/task.md', 'Help with analyzer.')
create = run_cli(f'task create {ambiguous_task}')
clarify_id = create.stdout.strip().splitlines()[-1]
run_cli(f'task status {clarify_id}')
run_cli(f'''task clarify {clarify_id} "Please explain analyzer and include implications for meridian_aux/src/meridian_aux/charts/transformed.py."''')
run_cli(f'task show {clarify_id} --evidence-summary')


## Active-task redirection

In [ ]:
run_cli(f'''task redirect {clarify_id} "Switch focus to compatibility impact for src/meridian_aux/dashboard/nordic_client.py" --new-cycle''')
run_cli(f'task status {clarify_id}')


## Deliver one task, then create a follow-on

In [ ]:
deliver_task = write_demo_task('tasks/review_delivery_seed/task.md', '''
How do I use src/meridian_aux/charts/transformed.py from an existing Meridian object?
Provide a concise snippet.
''')
seed = run_cli(f'task create {deliver_task}')
seed_id = seed.stdout.strip().splitlines()[-1]
run_cli(f'task run {seed_id} --to-gate')
for item in json.loads(run_cli(f'review queue --task-id {seed_id}').stdout):
    run_cli(f"review decide {item['review_id']} approve")
run_cli(f'task run {seed_id} --through-delivery')
follow = write_demo_task('tasks/follow_on_from_seed/task.md', '''
Given the previous transformed.py guidance, add guardrails for coordinate validation
with meridian.data.input_data.InputData.
''')
run_cli(f'task follow-on {seed_id} {follow}')


## Prototype isolation + learning/exemplar blocking

In [ ]:
delivery_root = WORKSPACE / 'tasks' / seed_id / 'deliveries' / 'prototype' / 'D01'
print('prototype delivery files:')
for path in sorted(delivery_root.iterdir()):
    print('-', path.name)

run_cli(f'learning nominate {seed_id}', check=False)
run_cli(f'exemplar nominate {seed_id}', check=False)
